### Procesamiento de Lenguaje Natural I
# **Desafío 1**

## Cristian Aballay
## SIU A2301



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [2]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [4]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [5]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [6]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [7]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [8]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [9]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [10]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [11]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [12]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [13]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [14]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [15]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [16]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828])

Obtenemos los 5 documentos más similares:

In [17]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [18]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [19]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [20]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [21]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [22]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**


**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.


In [ ]:
# tomamos 5 documentos al azar
#import random
#unique_numbers = random.sample(range(0, len(newsgroups_train.data)), 5)
#print(unique_numbers)

[3006, 4231, 8203, 5375, 2509]


In [23]:
# Seteamos los números únicos para no perder los indices de los documentos seleccionados y analizarlos
unique_numbers = [6628, 2683, 9349, 4869, 7263]
documents = []
for idx in unique_numbers:
  print(f'Índice del documento: {idx}')
  # Vemos el contenido del documento a analizar
  print(f'Clase del documento: {newsgroups_train.target_names[y_train[idx]]}')
  print(f'Número de palabras en el documento: {len(newsgroups_train.data[idx].split())}')
  print(f'Contenido del documento:\n{newsgroups_train.data[idx]}')
  # Medimos la similitud coseno
  cossim = cosine_similarity(X_train[idx], X_train)[0]
  # Obtenemos los 5 documentos más similares
  mostsim = np.argsort(cossim)[::-1][1:6]
  print(f'Similitud coseno 5 mas similares: {mostsim}')
  documents.append((idx, mostsim))
  for i in mostsim:
    print(f'* Índice del documento similar: {i}')
    print(f'Similitud coseno: {cossim[i]}')
    print(f'Clase del documento similar: {newsgroups_train.target_names[y_train[i]]}')
    print(f'Número de palabras en el documento similar: {len(newsgroups_train.data[i].split())}')
    print(f'Contenido del documento similar:\n{newsgroups_train.data[i]}')
  print('---' * 20)

Índice del documento: 6628
Clase del documento: comp.sys.mac.hardware
Número de palabras en el documento: 34
Contenido del documento:
Are there any PDS expansion cards out there that specifically take
advantage of the LCIII's 32 bit data path and 25MHz clock speed? If
they exist, are they significantly faster than the LC/LCII versions?
Similitud coseno 5 mas similares: [10146  2597  6647  5569  2770]
* Índice del documento similar: 10146
Similitud coseno: 0.34184980011468274
Clase del documento similar: comp.sys.mac.hardware
Número de palabras en el documento similar: 119
Contenido del documento similar:
Second Wave makes NuBus card cages that work on the PDS slots of at
least three Macs: the SE/30, IIsi and Centris 610. They have not, to
my knowledge, announced such a device for the LCII, but they could
make one, technologically.

The PDS card that goes to the cage simply needs the NuBus controller
circuitry present on NuBus Macs.

Why, though, does anyone care about this? dgr has a t

**Analisis**

De acuerdo a lo analizado en los distintos documentos, podemos ver que la similitud coseno asigno perfectamente las clases de documentos con los documentos similares. Es el caso del documento id 6628, de tipo hadware y donde los documentos similares tienen aproximadamente similitud coseno de 0.2, bajo pero que asimila bastante bien. 

A pesar de esto, existen casos en que a el calculo de similitud coseno es alto, los documentos similares asociados son de otra clase . Es el caso del documento con id 9349 de clase graphics donde los documentos similares son de otro clase pero tienen similitud coseno de 0.5.

Esta diferencia podria deberse por 2 motivos. Por un lado quizas las palabras vectorizadas para una clase en particular son genericas y aplican para varias clases, mientras que otras (como en laso de palabras en hardware o space) son mas especificas y se utilizan en determinados escenarios.

Por lo tanto el calculo de similitud coseno se tiene que acompañar con un analisis del contexto del documento.


**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

In [24]:
from sklearn.neighbors import KNeighborsClassifier
neigh = KNeighborsClassifier(n_neighbors=20)
neigh.fit(X_train, y_train)
y_pred_knn = neigh.predict(X_test)
f1_score_knn = f1_score(y_test, y_pred_knn, average='macro')
print(f'F1 Score KNN: {f1_score_knn}')

F1 Score KNN: 0.05799172287024525


In [25]:
#unique_numbers_test = random.sample(range(0, len(newsgroups_test.data)), 5)
unique_numbers_test = [2757, 4657, 4776, 463, 7032]
for idx in unique_numbers_test:
  print(f'Índice del documento: {idx}')
  # Vemos el contenido del documento a analizar
  print(f'Clase predicha: {newsgroups_test.target_names[y_pred_knn[idx]]}')
  # Medimos la similitud coseno
  cossim = cosine_similarity(X_test[idx], X_train)[0]
  # Obtenemos los 5 documentos más similares
  mostsim = np.argsort(cossim)[::-1][1:6]
  print(f'Similitud coseno 5 mas similares: {mostsim}')
  for i in mostsim:
    print(f'{i} - Similitud coseno: {cossim[i]}- Clase del documento similar: {newsgroups_train.target_names[y_train[i]]}')
  print('---' * 20)

Índice del documento: 2757
Clase predicha: sci.electronics
Similitud coseno 5 mas similares: [8150 9623 6894 1292 6552]
8150 - Similitud coseno: 0.17959950624726465- Clase del documento similar: rec.autos
9623 - Similitud coseno: 0.1763230908413336- Clase del documento similar: talk.politics.mideast
6894 - Similitud coseno: 0.17467486630221618- Clase del documento similar: talk.politics.guns
1292 - Similitud coseno: 0.1637397026312035- Clase del documento similar: talk.politics.mideast
6552 - Similitud coseno: 0.15443468886997153- Clase del documento similar: talk.religion.misc
------------------------------------------------------------
Índice del documento: 4657
Clase predicha: comp.os.ms-windows.misc
Similitud coseno 5 mas similares: [8144 5799 5405 3364 9208]
8144 - Similitud coseno: 0.24873004903595491- Clase del documento similar: comp.windows.x
5799 - Similitud coseno: 0.2097401969377558- Clase del documento similar: comp.graphics
5405 - Similitud coseno: 0.19007191622392827- Cl

**Analisis**

Al entrenar el modelo con KNN y predecir el target correspondiente a los documentos de tests, se tomaron 5 documentos random de test para analizar la similaridad coseno frente a los documentos de train. Podemos ver que en todos los casos la similaridad es baja y no concuerda con las clases de los documentos "mas parecidos". Esto nos dice que el modelo predice muy mal, tal como se puede corroborar en el calculo de f1 score.


**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.


Para buscar los mejores hiperparametros y obtener el mejor scoring (f1 macro, en este caso) usaremos optuna, definiendo los limites en los parametros mas importantes tanto para nuestro vectorizador como para el modelo.

In [97]:
import optuna 
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

def dt_objective_multinominal(trial):
    max_features = trial.suggest_int("max_features", 1000, 10000)
    max_df = trial.suggest_float("max_df", 0.7, 1.0)
    min_df = trial.suggest_int("min_df", 1, 5)
    sublinear_tf = trial.suggest_categorical("sublinear_tf", [True, False])
    use_idf = trial.suggest_categorical("use_idf", [True, False])
    alpha = trial.suggest_float("alpha", 1e-2, 1.0, log=True)
    fit_prior = trial.suggest_categorical("fit_prior", [True, False])

    tfidfvect = TfidfVectorizer(max_features=max_features,max_df=max_df, min_df=min_df, sublinear_tf=sublinear_tf, use_idf=use_idf,stop_words="english")
    
    clf = MultinomialNB(alpha=alpha, fit_prior=fit_prior)
    X_train = tfidfvect.fit_transform(newsgroups_train.data)
    score = cross_val_score(clf, X_train, newsgroups_train.target, cv=5, n_jobs=-1, scoring='f1_macro')
    return score.mean()

def dt_objective_complement(trial):
    max_features = trial.suggest_int("max_features", 1000, 10000)
    max_df = trial.suggest_float("max_df", 0.7, 1.0)
    min_df = trial.suggest_int("min_df", 1, 5)
    sublinear_tf = trial.suggest_categorical("sublinear_tf", [True, False])
    use_idf = trial.suggest_categorical("use_idf", [True, False])
    alpha = trial.suggest_float("alpha", 1e-2, 1.0, log=True)
    fit_prior = trial.suggest_categorical("fit_prior", [True, False])

    tfidfvect = TfidfVectorizer(max_features=max_features,max_df=max_df, min_df=min_df, sublinear_tf=sublinear_tf, use_idf=use_idf,stop_words="english")
    
    clf = ComplementNB(alpha=alpha, fit_prior=fit_prior)
    X_train = tfidfvect.fit_transform(newsgroups_train.data)
    score = cross_val_score(clf, X_train, newsgroups_train.target, cv=5, n_jobs=-1, scoring='f1_macro')
    return score.mean()

In [98]:
study = optuna.create_study(direction="maximize")
study.optimize(dt_objective_multinominal, n_trials=20)

print("Mejores parámetros: ", study.best_params)
print("Mejor puntuación:", study.best_value)

[I 2026-05-08 15:07:56,271] A new study created in memory with name: no-name-6d991345-be50-4fa6-b214-20e9177775d0
[I 2026-05-08 15:08:13,136] Trial 0 finished with value: 0.7039111107559684 and parameters: {'max_features': 8491, 'max_df': 0.9191193520278557, 'min_df': 3, 'sublinear_tf': True, 'use_idf': True, 'alpha': 0.34320542305165214, 'fit_prior': True}. Best is trial 0 with value: 0.7039111107559684.
[I 2026-05-08 15:08:17,939] Trial 1 finished with value: 0.7039805140441256 and parameters: {'max_features': 5866, 'max_df': 0.9334051507119822, 'min_df': 5, 'sublinear_tf': False, 'use_idf': True, 'alpha': 0.1163442692977508, 'fit_prior': True}. Best is trial 1 with value: 0.7039805140441256.
[I 2026-05-08 15:08:22,551] Trial 2 finished with value: 0.7029995496017414 and parameters: {'max_features': 9343, 'max_df': 0.7402265760922714, 'min_df': 3, 'sublinear_tf': False, 'use_idf': True, 'alpha': 0.49769905744024256, 'fit_prior': True}. Best is trial 1 with value: 0.7039805140441256.


Mejores parámetros:  {'max_features': 9869, 'max_df': 0.8686888007564195, 'min_df': 2, 'sublinear_tf': False, 'use_idf': True, 'alpha': 0.04797696165807192, 'fit_prior': False}
Mejor puntuación: 0.7317277239951163


In [99]:
tfidfvect = TfidfVectorizer(max_features=study.best_params['max_features'], max_df=study.best_params['max_df'], min_df=study.best_params['min_df'], 
                            sublinear_tf=study.best_params['sublinear_tf'], stop_words="english", use_idf=study.best_params['use_idf'])
X_train = tfidfvect.fit_transform(newsgroups_train.data)

y_train = newsgroups_train.target
clf = MultinomialNB(alpha=study.best_params['alpha'], fit_prior=study.best_params['fit_prior'])
clf.fit(X_train, y_train)

X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred_multinomial =  clf.predict(X_test)
f1_score_multinomial = f1_score(y_test, y_pred_multinomial, average='macro')
print(f'F1 Score MultinomialNB: {f1_score_multinomial}')

F1 Score MultinomialNB: 0.665979170486523


In [100]:
study = optuna.create_study(direction="maximize")
study.optimize(dt_objective_complement, n_trials=20)

print("Mejores parámetros: ", study.best_params)
print("Mejor puntuación:", study.best_value)

[I 2026-05-08 15:09:16,030] A new study created in memory with name: no-name-a59ed041-2d29-4c32-b6f8-737517f5a358
[I 2026-05-08 15:09:17,621] Trial 0 finished with value: 0.6859804173376288 and parameters: {'max_features': 6002, 'max_df': 0.8774665037106714, 'min_df': 5, 'sublinear_tf': True, 'use_idf': False, 'alpha': 0.4855835982167263, 'fit_prior': True}. Best is trial 0 with value: 0.6859804173376288.
[I 2026-05-08 15:09:19,171] Trial 1 finished with value: 0.7080219702965593 and parameters: {'max_features': 9574, 'max_df': 0.905702429349648, 'min_df': 5, 'sublinear_tf': False, 'use_idf': False, 'alpha': 0.9333954128259772, 'fit_prior': False}. Best is trial 1 with value: 0.7080219702965593.
[I 2026-05-08 15:09:20,797] Trial 2 finished with value: 0.6709370968542638 and parameters: {'max_features': 4724, 'max_df': 0.8966337443694758, 'min_df': 4, 'sublinear_tf': False, 'use_idf': True, 'alpha': 0.016802230581423338, 'fit_prior': False}. Best is trial 1 with value: 0.708021970296559

Mejores parámetros:  {'max_features': 9818, 'max_df': 0.9333683751257971, 'min_df': 1, 'sublinear_tf': True, 'use_idf': False, 'alpha': 0.03253601604602536, 'fit_prior': True}
Mejor puntuación: 0.7096047596491569


In [102]:
tfidfvect = TfidfVectorizer(max_features=study.best_params['max_features'], max_df=study.best_params['max_df'], min_df=study.best_params['min_df'], 
                            sublinear_tf=study.best_params['sublinear_tf'], stop_words="english", use_idf=study.best_params['use_idf'])
X_train = tfidfvect.fit_transform(newsgroups_train.data)

y_train = newsgroups_train.target
cnb = ComplementNB(alpha=study.best_params['alpha'], fit_prior=study.best_params['fit_prior'])
cnb.fit(X_train, y_train)
X_test = tfidfvect.transform(newsgroups_test.data)
y_pred_complement = cnb.predict(X_test)
f1_score_complement = f1_score(y_test, y_pred_complement, average='macro')

In [103]:
print(f'F1 Score MultinomialNB: {f1_score_multinomial}')
print(f'F1 Score ComplementNB: {f1_score_complement}')

F1 Score MultinomialNB: 0.665979170486523
F1 Score ComplementNB: 0.6586928683371189


**Analisis**

Se probaron distintas configuraciones de hiperparametros y los que mejor resultado dieron fueron las siguientes:

Para definir los modelos de clasificacion en principio se instancia el vectorizador definiendo algunos hiperparametros para clasificar mejor segun optuna. Se define un limite maximo y minimo a cantidad de frecuencia de aparicion, ignorando aquellas que superen el ~90% e inferior a ~3 apariciones. Ademas se define sublinear_tf en True para MultinominalNB y False para ComplementNB que palabras que aparecen con mucha frecuencia dominen o no la puntuacion final.

En cuanto a los modelos, tanto para Multinominal como para Complement se definieron hiperparametros: alpha de ~0.03 para el suavizado para que el modelo confie más en las frecuencias reales de los datos de entrenamiento. Y fit_prior para que el modelo trate a todas las clases con la misma importancia inicial

Podemos ver que F1 score para ambos modelos es muy parecido, mejorando leventente con respecto al modelo inicial.


**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.

In [80]:
words = ['people', 'car', 'space', 'money', 'speed']
feature_names = tfidfvect.get_feature_names_out()
for word in words:
  print(f'Palabra: {word}')
  # Medimos la similitud coseno
  cossim = cosine_similarity(X_train.T[tfidfvect.vocabulary_[word]], X_train.T)[0]
  # Obtenemos los 5 documentos más similares
  mostsim = np.argsort(cossim)[::-1][1:6]
  print(f'Similitud coseno 5 mas similares: {mostsim}')
  for i in mostsim:
    print(f'{feature_names[i]} - Similitud coseno: {cossim[i]}')
  print('---' * 20)

Palabra: people
Similitud coseno 5 mas similares: [ 8622 23881 13831 11370 14645]
don - Similitud coseno: 0.24909053589612987
think - Similitud coseno: 0.2395716888268954
just - Similitud coseno: 0.1976059497223518
government - Similitud coseno: 0.19250751473798997
like - Similitud coseno: 0.19202109004195775
------------------------------------------------------------
Palabra: car
Similitud coseno 5 mas similares: [ 5506  7630  5992 17680  9352]
cars - Similitud coseno: 0.1956588714084559
dealer - Similitud coseno: 0.18318691267502366
civic - Similitud coseno: 0.17190304721528865
owner - Similitud coseno: 0.1613757813202762
engine - Similitud coseno: 0.1440342708140713
------------------------------------------------------------
Palabra: space
Similitud coseno 5 mas similares: [16554 21865 14353 13766 22749]
nasa - Similitud coseno: 0.2624500703507634
shuttle - Similitud coseno: 0.2155521108016394
launch - Similitud coseno: 0.18381022038250316
jsc - Similitud coseno: 0.171405981320258

**Analisis**

Cuando se realiza la transpuesta de documentos x palabras, y realizar el analisis de similaridad por palabras, podemos notar dos cosas. Por un lado que la similitud coseno es relativamente bajo en todos los casos (menor a 25%). Por otro lado que las palabras asociadas cuando son muy especificas (como por ejemplo 'space') tienen relacion con su grupo de similaridad, mientras que para parabras mas genericas (como 'people') en general no tienen nada que ver con el resto. Lo cual suena logico si pensamos que la similaridad se obtiene a partir de los documentos asociados, y si tenemos en cuenta que palabras tecnicas o especificas en presentan en determinados contextos coincidan en documentos similares. Caso contrario para palabras genericas, es probable que tenga apariciones en cualquier clase de documento.